In [ ]:
# GOOGLE COLAB SETUP
!pip install -q ipywidgets

# Part 1 — The Quant Math Toolbox



We are not going to derive Brownian motion, PDE theory or other complex things here. The goal is to introduce the ideas and see how they appear in code.

## 0. Setup

NumPy gives us fast arrays and linear algebra. SciPy adds things like probability distributions, integration and differential-equation solvers. Matplotlib lets us see what the mathematics is doing.

NumPy's core model is the multidimensional array, with vectorized operations and linear-algebra tools built around it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider

rng = np.random.default_rng(42)

## 1. NumPy: prices become vectors

In quantitative finance, we constantly operate on entire vectors of observations or scenarios at once.

> Most of ML and Data Analytics classes at UB use numpy, e.g. MTH337, CSE474 (and many more).

Here is a tiny price series:

In [ ]:
prices = np.array([100, 102, 101, 105, 107, 106])
prices

A simple return is the percentage change from one price to the next.

For two consecutive prices,

$$
R_t = \frac{P_t}{P_{t-1}} - 1
$$

Before running the next cell, ask:

**What shape do you expect `returns` to have?**

In [ ]:
# ===== LIVE CODING 1 =====

We can apply one operation to an entire vector of data at once.

For example, converting returns to percentages can be done in one line:

In [ ]:
returns * 100

The conceptual Python-loop version would be:

In [ ]:
[r * 100 for r in returns]

We do not need to benchmark them here. The main idea is to start thinking in arrays instead of element-by-element loops.

Now visualize the price series:

In [ ]:
plt.plot(prices, marker="o")
plt.xlabel("Time")
plt.ylabel("Price")
plt.title("A Tiny Price Series")
plt.show()

## 2. Linear algebra: a portfolio is a dot product
> Related classes: MTH309, MTH420

This is a useful introductory linear-algebra example because it immediately answers:

**Why do quants care about vectors?**

Suppose three assets have returns:

In [ ]:
asset_returns = np.array([
    0.08,   # asset A
    0.03,   # asset B
   -0.02    # asset C
])

weights = np.array([
    0.50,
    0.30,
    0.20
])

The portfolio return can be written as

$$
R_p = w^\top r
$$

In Python, `@` performs the dot product / matrix multiplication.

In [ ]:
# ===== LIVE CODING 2 =====

This is equivalent to

$$
0.50(0.08) + 0.30(0.03) + 0.20(-0.02)
$$

A surprising amount of finance eventually becomes some version of this operation.

### Make it interactive

The widget below lets us change the first two portfolio weights.

The third weight is whatever remains so that the weights sum to 1.

In [ ]:
@interact(
    w_a=FloatSlider(min=0, max=1, step=0.05, value=0.5),
    w_b=FloatSlider(min=0, max=1, step=0.05, value=0.3)
)
def portfolio_demo(w_a, w_b):
    w_c = 1 - w_a - w_b

    if w_c < 0:
        print("Weights must sum to at most 1.")
        return

    weights = np.array([w_a, w_b, w_c])
    r = np.array([0.08, 0.03, -0.02])

    print("Weights:", np.round(weights, 2))
    print(f"Portfolio return: {weights @ r:.2%}")

### Covariance example

We will not explain covariance matrices deeply here.

The important point is that a covariance matrix describes how assets move individually and together.

Portfolio volatility can be written as

$$
\sigma_p = \sqrt{w^\top \Sigma w}
$$


In [ ]:
cov = np.array([
    [0.04, 0.01],
    [0.01, 0.09]
])

w = np.array([0.6, 0.4])

In [ ]:
# ===== LIVE CODING 3 =====

## 3. Probability: uncertainty becomes a distribution

> Related classes: STA301 (MTH411), STA302 (MTH412), and many others

So far our returns were already known.

Finance becomes interesting because future returns are not known.

A probability distribution gives us a way to describe uncertain outcomes.

Start with simulated observations from a standard normal distribution:

In [ ]:
samples = rng.normal(loc=0, scale=1, size=10_000)

plt.hist(samples, bins=50, density=True, alpha=0.6)

x = np.linspace(-4, 4, 300)
plt.plot(x, stats.norm.pdf(x))

plt.title("Random Samples vs. Normal Distribution")
plt.xlabel("x")
plt.ylabel("Density")
plt.show()

The histogram is generated from random samples.

The smooth curve is the theoretical normal probability density.

With enough samples, the histogram starts to resemble the theoretical distribution.

Now we can use SciPy for the cumulative distribution function, or CDF.

In [ ]:
# ===== LIVE CODING 4 =====

The result is approximately

$$
0.0228
$$

or about 2.28%.

Questions like

> What is the probability of losing more than X?

are distribution questions.

The CDF is especially important for Black–Scholes because the formula later contains terms like $N(d_1)$ and $N(d_2)$.

### Interactive distribution

The normal distribution has two important parameters:

- $\mu$ controls the center.
- $\sigma$ controls the spread.

You can immediately see what volatility means geometrically by changing $\sigma$.

In [ ]:
@interact(
    mu=(-1.00, 1.00, 0.01),
    sigma=(0.10, 1.00, 0.01)
)
def normal_demo(mu=0.05, sigma=0.20):
    x = np.linspace(-2, 2, 400)
    y = stats.norm.pdf(x, loc=mu, scale=sigma)

    plt.plot(x, y)
    plt.axvline(mu)

    plt.xlim(-2, 2)
    plt.ylim(0, 5.0)

    plt.xlabel("Return")
    plt.ylabel("Density")
    plt.title(f"Normal distribution: μ={mu:.2f}, σ={sigma:.2f}")
    plt.show()

## 4. Monte Carlo: turn probability into computation

> Related classes: MTH446, MTH443, STA411

Suppose an asset is \$100 today.

One year from now, let's model its return as random.

What might the distribution of future prices look like?

In [ ]:
S0 = 100
mu = 0.08
sigma = 0.20

returns = rng.normal(mu, sigma, size=10_000)
future_prices = S0 * (1 + returns)

plt.hist(future_prices, bins=60)
plt.axvline(future_prices.mean(), linestyle="--")
plt.xlabel("Future price")
plt.ylabel("Count")
plt.title("10,000 Possible Futures")
plt.show()

The conceptual Monte Carlo idea is

$$
\mathbb{E}[f(X)]
\approx
\frac{1}{N}\sum_{i=1}^{N} f(X_i)
$$

In plain language:

> Simulate many possible worlds, compute what you care about in each world, then average.

In [ ]:
print("Estimated future price:", future_prices.mean())
print("Probability price is below 80:", np.mean(future_prices < 80))

The second line works because `future_prices < 80` creates `True` and `False` values.

NumPy treats `True` like 1 and `False` like 0 when taking the mean.

So the result is the fraction of simulations below 80.

### Interactive Monte Carlo

Changing the number of simulations lets us see how the estimate behaves as we use more samples.

## 5. Stochastic processes: not one random number, but a random path

> Related classes: MTH446, MTH443, STA411

Monte Carlo gave us one random value at maturity.

But asset prices move continuously through time.

So now we need a sequence of random changes.

In [ ]:
@interact(
    n=(100, 50_000, 100),
    sigma=(0.05, 0.60, 0.05)
)
def mc_demo(n=1000, sigma=0.20):
    samples = rng.normal(0.08, sigma, size=n)
    ST = 100 * (1 + samples)

    plt.xlim(40, 180)

    plt.hist(ST, bins=40)
    plt.axvline(ST.mean(), linestyle="--")
    plt.xlabel("Future price")
    plt.title(f"{n:,} simulated outcomes")
    plt.show()

    print(f"Estimated E[S_T]: {ST.mean():.2f}")

## Brownian motion


Let

$$
T = 1
$$

year, and split it into 252 small steps.

Then

$$
\Delta t = \frac{T}{252}
$$

A Brownian increment can be generated as

$$
\Delta W = \sqrt{\Delta t}\,Z
$$

where

$$
Z \sim N(0,1)
$$

In [ ]:
T = 1
steps = 252
dt = T / steps

In [ ]:
# ===== LIVE CODING 5 =====

The three key pieces are:

- `rng.normal(...)` creates random shocks.
- `np.sqrt(dt)` scales those shocks with time.
- `np.cumsum(...)` turns increments into a path.

### Multiple Brownian paths

A single Brownian path is not a prediction.

If we repeat the experiment, we get a different path each time.

In [ ]:
n_paths = 20

dW = np.sqrt(dt) * rng.normal(size=(n_paths, steps))
W = np.cumsum(dW, axis=1)

W = np.column_stack([np.zeros(n_paths), W])

for path in W:
    plt.plot(t, path, alpha=0.6)

plt.xlabel("Time")
plt.ylabel("W(t)")
plt.title("Possible Brownian Paths")
plt.show()

## 6. Geometric Brownian motion: our first toy price model

> Related classes: MTH446, MTH443, STA411

Now connect Brownian motion directly to finance.

Without deriving it, we can write geometric Brownian motion as

$$
dS_t = \mu S_t\,dt + \sigma S_t\,dW_t
$$

There is a deterministic part (drift), and a random part (volatility)

Here:

- $S_t$ is the asset price
- $\mu$ is the drift
- $\sigma$ is volatility
- $dW_t$ is the Brownian shock

In [ ]:
S0 = 100
mu = 0.08
sigma = 0.20

Z = rng.normal(size=(20, steps))

In [ ]:
# ===== LIVE CODING 6 =====

In [ ]:
S = np.column_stack([np.full(20, S0), S])

for path in S:
    plt.plot(t, path, alpha=0.7)

plt.xlabel("Time")
plt.ylabel("Price")
plt.title("Geometric Brownian Motion")
plt.show()

Several ideas from earlier now appear together:

- normal random variables
- volatility
- time
- cumulative random shocks
- exponentials

Geometric Brownian motion is one of the central assumptions behind the Black–Scholes model.

### Interactive GBM explorer

Ask:

**What happens if we double volatility?**

Then change $\sigma$ and look at how the paths spread out.

In [ ]:
@interact(
    mu=(-0.10, 0.30, 0.02),
    sigma=(0.05, 0.80, 0.05),
    paths=(1, 100, 5)
)
def gbm_demo(mu=0.08, sigma=0.20, paths=10):
    Z = rng.normal(size=(paths, steps))

    increments = (
        (mu - 0.5 * sigma**2) * dt
        + sigma * np.sqrt(dt) * Z
    )

    S = S0 * np.exp(np.cumsum(increments, axis=1))
    S = np.column_stack([np.full(paths, S0), S])

    for path in S:
        plt.plot(t, path, alpha=0.65)

    plt.axhline(S0, linestyle="--")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.title("GBM: drift vs. volatility")
    plt.show()

## 7. ODEs

> Related classes: MTH306, MTH418

You do not need to understand differential equations deeply here.

The main goal is to recognize what they look like and know that Python can solve them numerically.

Use the simplest possible financial differential equation:

$$
\frac{dV}{dt} = rV
$$

This represents continuously compounded growth.

We know the analytical solution:

$$
V(t) = V_0 e^{rt}
$$

Now solve the same equation numerically.

In [ ]:
r = 0.05
V0 = 100

def growth(t, V):
    return r * V

In [ ]:
# ===== LIVE CODING 7 =====

In [ ]:
t_grid = np.linspace(0, 10, 200)

numerical = solution.sol(t_grid)[0]
exact = V0 * np.exp(r * t_grid)

plt.plot(t_grid, numerical, label="Numerical")
plt.plot(t_grid, exact, "--", label="Exact")

plt.xlabel("Years")
plt.ylabel("Value")
plt.legend()
plt.title("Solving an ODE Numerically")
plt.show()

`solve_ivp` numerically solves initial-value differential equations of the form

$$
\frac{dy}{dt} = f(t,y)
$$


For Black–Scholes, this matters because option pricing can also be viewed through differential equations and numerical methods.

## 8. PDEs

> Related classes: MTH306, MTH418

So far, we have mostly worked with equations that describe how quantities change with respect to one variable.

A partial differential equation (PDE) describes a function that depends on multiple variables and how it changes with respect to each of them.

PDEs appear throughout quantitative finance.

One famous example is the Black–Scholes PDE:

$$
\frac{\partial V}{\partial t}
+
\frac{1}{2}\sigma^2 S^2
\frac{\partial^2 V}{\partial S^2}
+
rS\frac{\partial V}{\partial S}
-
rV
=
0
$$

We are not going to solve PDEs in this workshop.

For now, the important idea is simply that many problems in quantitative finance can be expressed using differential equations like this one.

Later, we will work directly with the Black–Scholes pricing formula rather than solving this PDE.

# Part 2 — Options Pricing

> Related classes: MTH458

The goal is to start with the simplest possible option payoff, price it using a tree, add more and more time steps, and see how this leads naturally to the Black–Scholes formula.

## 0. Setup

We will use the same tools as Part 1.

## 1. Options: turning a price into a payoff

A European option gives us a payoff at maturity.

We will use stock price at maturity:

$$S_T$$
and strike price:
$$K$$

For a European call,

$$
C = \max(S_T-K,0)
$$

For a European put,

$$
P = \max(K-S_T,0)
$$

The max is the entire option logic.

In [ ]:
K = 100

S_T = np.linspace(50, 150, 400)

call_payoff = np.maximum(S_T - K, 0)
put_payoff = np.maximum(K - S_T, 0)


plt.plot(S_T, call_payoff, label="Call payoff")
plt.plot(S_T, put_payoff, label="Put payoff")

plt.axvline(K, linestyle="--", label="Strike K")

plt.xlabel("Stock price at maturity $S_T$")
plt.ylabel("Option payoff")
plt.title("European Option Payoffs")
plt.legend()
plt.show()

The important point is at $K$.

Below the strike, the call is worth zero at maturity.

Above the strike, every additional dollar in the stock creates one additional dollar of call payoff.

The put does the opposite.

## Make the strike interactive

In [ ]:
@interact(K=FloatSlider(min=60, max=140, step=5, value=100))
def payoff_demo(K):
    S_T = np.linspace(40, 160, 400)

    call = np.maximum(S_T - K, 0)
    put = np.maximum(K - S_T, 0)

    plt.plot(S_T, call, label="Call")
    plt.plot(S_T, put, label="Put")

    plt.axvline(K, linestyle="--", label=f"K = {K}")
    plt.xlabel("Stock price at maturity")
    plt.ylabel("Payoff")

    plt.title("Moving the Strike Price")

    plt.legend()
    plt.show()

Important distinction:

This graph shows the payoff at maturity, not what the option is worth today.

Pricing the option today is the next problem.

## 2. One-period binomial model

Suppose the stock is worth

$$
S_0 = 100
$$

today.

One period from now, imagine only two possibilities:

$$
S_u = S_0u
$$

or

$$
S_d = S_0d
$$

For example:

In [ ]:
S0 = 100
K = 100

u = 1.20
d = 0.90

r = 0.05
T = 1

S_up = S0 * u
S_down = S0 * d

print("Up state:", S_up)
print("Down state:", S_down)

Visualize the tree:

In [ ]:
plt.plot([0, 1], [S0, S_up], marker="o")
plt.plot([0, 1], [S0, S_down], marker="o")

plt.axhline(K, linestyle="--", label="Strike K")

plt.xticks([0, 1], ["Today", "Maturity"])
plt.ylabel("Stock price")
plt.title("One-Period Binomial Tree")
plt.legend()
plt.show()

Now compute the call payoff in each possible world:

$$
C_u = \max(S_u-K,0)
$$

$$
C_d = \max(S_d-K,0)
$$

In [ ]:
C_up = max(S_up - K, 0)
C_down = max(S_down - K, 0)

P_up = max(K - S_up, 0)
P_down = max(K - S_down, 0)

print("Call:", C_up, C_down)
print("Put:", P_up, P_down)

But how do we turn future payoffs into a price today?

### Risk-neutral probability
Instead of trying to predict the stock's real probability of going up, option pricing uses special probabilities chosen so that the stock grows at the risk-free rate on average.

Let $p$ be the risk-neutral probability of an up move.

Let $q$ be the risk-neutral probability of a down move.

$$
p =
\frac{e^{rT}-d}{u-d}
$$

$$
q = 1-p
$$

Then the option price is the discounted expected payoff:

$$
C_0 =
e^{-rT}
\left[
pC_u + qC_d
\right]
$$

Notice something unusual:

There is no expected stock return in this formula.

In [ ]:
p = (np.exp(r * T) - d) / (u - d)
q = 1 - p

call_price = np.exp(-r * T) * ( p * C_up + q * C_down )
put_price = np.exp(-r * T) * ( p * P_up + q * P_down )

print(f"Risk-neutral up probability p: {p:.3f}")
print(f"Risk-neutral down probability q: {q:.3f}")
print(f"Call price today: ${call_price:.2f}")
print(f"Put price today: ${put_price:.2f}")

## 3. Trinomial model: what if there are three possibilities?
The binomial model says that during one step the stock can move:

- up
- down

A trinomial model allows:

- up
- approximately unchanged
- down

Conceptually, nothing changes.

Instead of

$$
V = e^{-r\Delta t}
\left[
pV_u + qV_d
\right]
$$

we have

$$
V = e^{-r\Delta t}
\left[
p_uV_u+p_mV_m+p_dV_d
\right]
$$

with

$$
p_u+p_m+p_d=1.
$$

There are different ways to calibrate a trinomial tree.

We will use one common choice without deriving it.

In [ ]:
S0 = 100
K = 100
r = 0.05
sigma = 0.20
T = 1

dt = T

u3 = np.exp(sigma * np.sqrt(2 * dt))
d3 = 1 / u3

a = np.exp(r * dt / 2)
b = np.exp(sigma * np.sqrt(dt / 2))
c = 1 / b

p_up = ((a - c) / (b - c))**2
p_down = ((b - a) / (b - c))**2
p_middle = 1 - p_up - p_down

ST_tri = np.array([ S0 * u3, S0, S0 * d3 ])

probs_tri = np.array([ p_up, p_middle, p_down ])

call_payoffs_tri = np.maximum(ST_tri - K, 0)

print("Possible prices:", np.round(ST_tri, 2))
print("Probabilities:", np.round(probs_tri, 3))

In [ ]:
# ===== LIVE CODING 8 =====

The main idea is:
Create possible future states, calculate the payoff in each state, weight them using risk-neutral probabilities, and discount back.

Binomial and trinomial trees are two versions of that same idea.

## 4. Expanding the binomial model
One time step is obviously unrealistic.

So split the total time into $N$ periods.

If

$$
\Delta t = \frac{T}{N},
$$

then every individual step represents a smaller amount of time.

We will keep using explicit up and down multipliers:

$$
u > 1
$$

$$
d < 1
$$

For example, during each period we might allow the stock to move up by a factor of (1.10) or down by a factor of (0.95).

The risk-neutral probability for each period becomes

$$
p=
\frac{e^{r\Delta t}-d}{u-d}
$$

and

$$
q=1-p.
$$

In [ ]:
S0 = 100
K = 100
r = 0.05
T = 1

N = 5

u = 1.10
d = 0.95

dt = T / N

p = (np.exp(r * dt) - d) / (u - d)
q = 1 - p

print("dt:", dt)
print("u:", u)
print("d:", d)
print("p:", p)
print("q:", q)

The structure is exactly the same as the one-period model.

We have simply repeated the up/down experiment multiple times.

### Terminal stock prices

After $N$ periods, different combinations of up and down moves produce different terminal prices.

If there are $j$ up moves, then there are $N-j$ down moves.

Therefore,

$$
S_T =
S_0u^jd^{N-j}.
$$

In [ ]:
j = np.arange(N + 1)

ST = S0 * (u**j) * (d**(N - j))

ST

And now we already know how to turn those stock prices into option payoffs.

In [ ]:
# ===== LIVE CODING 9 =====

## 5. Backward induction
At maturity we know the option payoff.

So instead of trying to jump directly from today to maturity, work backwards through the tree.

At every node,

$$
V =
e^{-r\Delta t}
\left[
pV_u + qV_d
\right].
$$

One step backwards reduces $N+1$ possible option values to $N$.

Then $N$ becomes $N-1$.

Eventually only one number remains.

That number is today's option price.

In [ ]:
# ===== LIVE CODING 10 =====

This is one of the most important numerical ideas in derivatives pricing:

Start where the payoff is known and recursively work backwards.

### Put everything into a function

After doing the pieces live, package them together.

In [ ]:
def binomial_price(S0, K, r, T, N, u, d, kind="call"):
    dt = T / N

    p = (np.exp(r * dt) - d) / (u - d)
    q = 1 - p

    j = np.arange(N + 1)
    ST = S0 * (u**j) * (d**(N - j))

    if kind == "call":
        values = np.maximum(ST - K, 0)
    else:
        values = np.maximum(K - ST, 0)

    for step in range(N - 1, -1, -1):
        values = np.exp(-r * dt) * (
            p * values[1:] +
            q * values[:-1]
        )

    return values[0]

print(
    "Call:",
    binomial_price(
        100, 100, 0.05, 1, 5,
        1.10, 0.95, "call"
    )
)

print(
    "Put:",
    binomial_price(
        100, 100, 0.05, 1, 5,
        1.10, 0.95, "put"
    )
)

### Visualize a small tree

Large trees become unreadable very quickly, so use only four periods.

In [ ]:
N_plot = 4

u_plot = 1.10
d_plot = 0.95

tree = []

for i in range(N_plot + 1):
    j = np.arange(i + 1)

    prices = ( S0 * (u_plot**j) * (d_plot**(i - j)) )

    tree.append(prices)

for i in range(N_plot):
    for j in range(i + 1):
        plt.plot( [i, i + 1], [tree[i][j], tree[i + 1][j]], marker="o" )

        plt.plot( [i, i + 1], [tree[i][j], tree[i + 1][j + 1]], marker="o" )

plt.axhline(K, linestyle="--", label="Strike K")
plt.xlabel("Time step")
plt.ylabel("Stock price")
plt.title("Multi-Period Binomial Tree")
plt.legend()
plt.show()

A five-step or ten-step tree is manageable computationally.

But imagine doing this with hundreds, thousands, or infinitely many tiny time intervals.

That motivates a different way to describe stock-price uncertainty.

## 6. From discrete time to continuous time

The binomial model describes uncertainty using two explicit numbers:

- an up move $u$
- a down move $d$

Every period, the stock jumps from one branch of the tree to another.

But real markets do not wait until the end of a period and then suddenly jump.

Prices can change continuously.

If we make our time intervals shorter and shorter,

$$
\Delta t \rightarrow 0,
$$

the idea of individual tree branches becomes less useful.

Instead of describing uncertainty using a specific up move and down move, a continuous model needs a way to describe how much the stock tends to move.

That brings us to a new quantity.

## 7. Black–Scholes

Black–Scholes introduces volatility, written as

$$
\sigma
$$

Volatility measures the scale of the stock's price uncertainty.

A larger value of $\sigma$ means the stock price can move around more dramatically.

For example:

In [ ]:
sigma = 0.20

means we are modeling the stock with an annualized volatility of (20%).

In the continuous model, the stock follows risk-neutral geometric Brownian motion:

$$
dS_t = rS_t\,dt + \sigma S_t\,dW_t
$$

Compare this with the binomial tree.

The tree used:

- $u$ to describe an up move
- $d$ to describe a down move

The continuous model instead uses:

- $r$ for the risk-neutral drift
- $\sigma$ for the amount of randomness

The individual up/down branches have disappeared.

### Black–Scholes formula

For a European call with no dividends,

$$
C_0 = S_0 N(d_1) - K e^{-rT} N(d_2)
$$

where

$$
d_1 =
\frac{
\ln(S_0/K)
+
\left(r+\frac{1}{2}\sigma^2\right)T
}{
\sigma\sqrt{T}
}
$$

and

$$
d_2=d_1-\sigma\sqrt{T}.
$$

For a European put,

$$
P_0 = K e^{-rT} N(-d_2) - S_0 N(-d_1)
$$

Remember stats.norm.cdf from Part 1?

That is exactly the $N(\cdot)$ appearing here.

Our inputs are now:

In [ ]:
S0 = 100
K = 100

r = 0.05
sigma = 0.20

T = 1

In [ ]:
d1 = (
    np.log(S0 / K)
    + (r + 0.5 * sigma**2) * T
) / (sigma * np.sqrt(T))

d2 = d1 - sigma * np.sqrt(T)

call_bs = (
    S0 * stats.norm.cdf(d1)
    - K * np.exp(-r * T)
    * stats.norm.cdf(d2)
)

put_bs = (
    K * np.exp(-r * T)
    * stats.norm.cdf(-d2)
    - S0 * stats.norm.cdf(-d1)
)


print(f"d1 = {d1:.3f}")
print(f"d2 = {d2:.3f}")

print(
    f"Black-Scholes call: ${call_bs:.2f}"
)

print(
    f"Black-Scholes put:  ${put_bs:.2f}"
)

Now package the formula:

In [ ]:
def black_scholes(S, K, r, sigma, T):
    d1 = ( np.log(S / K) + (r + 0.5 * sigma**2) * T ) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call = ( S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2))
    put = ( K * np.exp(-r * T) * stats.norm.cdf(-d2) - S * stats.norm.cdf(-d1))

    return call, put

## 8. Graph Black–Scholes option values

Earlier we graphed the payoff at maturity.

Now graph the value of the option before maturity.

In [ ]:
S_grid = np.linspace(40, 160, 400)

call_values, put_values = black_scholes( S_grid, K=100, r=0.05, sigma=0.20, T=1 )

plt.plot( S_grid, call_values, label="Call value" )
plt.plot( S_grid, put_values, label="Put value" )

plt.axvline( 100, linestyle="--", label="Strike K" )

plt.xlabel("Stock price today")
plt.ylabel("Option value today")
plt.title("Black-Scholes Option Values")
plt.legend()
plt.show()

Compare this with the payoff graph from the beginning.

The lines are now curved rather than just having a sharp payoff kink.

### What does volatility actually do?

Now that we have introduced ($\sigma$), we can see its effect directly.

In [ ]:
# ===== LIVE CODING 11 =====

Higher volatility creates more uncertainty.

For an option, that extra uncertainty can be valuable because the downside payoff is limited while the upside can continue growing.

### Interactive Black–Scholes explorer

In [ ]:
@interact(
    sigma=(0.05, 0.80, 0.05),
    T=(0.10, 2.00, 0.10)
)
def bs_demo(sigma=0.20, T=1.0):
    S = np.linspace(40, 160, 400)

    call, put = black_scholes( S, K=100, r=0.05, sigma=sigma, T=T )

    plt.plot(S, call, label="Call")
    plt.plot(S, put, label="Put")

    plt.axvline( 100, linestyle="--" )

    plt.xlabel("Stock price")
    plt.ylabel("Option value")

    plt.title( f"Black-Scholes: σ={sigma:.2f}, T={T:.1f}" )

    plt.legend()
    plt.show()

This lets us see two ideas immediately:

- higher volatility generally makes European calls and puts more valuable,
- changing time to maturity changes how much uncertainty remains.

## 9. The whole story

We started with nothing more complicated than

$$
\max(S_T-K,0).
$$

Then:

1. A one-period binomial tree gave us two possible future prices.
2. Risk-neutral probabilities let us price those future payoffs.
3. A trinomial tree showed that we are not restricted to only two future states.
4. A multi-period binomial tree repeated the same pricing logic over many periods.
5. Moving toward continuous time made individual up and down moves less useful as a description.
6. Black–Scholes introduced volatility ($\sigma$) to describe continuous price uncertainty.
7. The Black–Scholes formula gave us a closed-form price for European calls and puts.

The key conceptual change is:

Binomial model

$$
u,\ d
$$

describe explicit discrete moves.

Black–Scholes

$$
\sigma
$$

describes continuous uncertainty.